<a href="https://colab.research.google.com/github/kiriakosgp/papadopoulos_av_analysis/blob/main/late_fusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
SPLIT_DIR = "/content/drive/MyDrive/splits (1)"
EMB_DIR = "/content/drive/MyDrive/transcript_embeddings"
BASELINE_JSON = "/content/drive/MyDrive/multimodal_baselines.json"
SAVE_PATH = "/content/drive/MyDrive/results/late_fusion_text_newsplits"

os.makedirs(SAVE_PATH, exist_ok=True)

In [ ]:
LABEL_MAP = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

In [ ]:
train_ids = set(pd.read_csv(os.path.join(SPLIT_DIR, "train.csv"))["video_id"].astype(str))

print("Train split size:", len(train_ids))

In [ ]:
with open(BASELINE_JSON, "r") as f:
    data = json.load(f)

records = []
for item in data:
    if item["transcript_pred"] is None:
        continue

    records.append({
        "video_id": str(item["video_id"]),
        "label": LABEL_MAP[item["transcript_pred"]]
    })

print("Total baseline records:", len(records))

In [ ]:
X_train = []
y_train = []
missing = 0

for r in records:
    vid = r["video_id"]

    if vid not in train_ids:
        continue

    emb_path = os.path.join(EMB_DIR, f"{vid}.npy")

    if not os.path.exists(emb_path):
        missing += 1
        continue

    emb = np.load(emb_path)
    X_train.append(emb)
    y_train.append(r["label"])

X_train = np.stack(X_train)
y_train = np.array(y_train)

print("Training samples:", X_train.shape[0])
print("Missing embeddings:", missing)

In [ ]:
class EmbeddingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = EmbeddingDataset(X_train, y_train)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

In [ ]:
class TextSentimentMLP(nn.Module):
    def __init__(self, input_dim, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.net(x)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AudioSentimentMLP(
    input_dim=X_train.shape[1],
    num_classes=3
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [ ]:
EPOCHS = 15

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(Xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss {total_loss/len(train_loader):.4f}")

Epoch 1 | Loss 1.0119
Epoch 2 | Loss 0.8815
Epoch 3 | Loss 0.7953
Epoch 4 | Loss 0.7080
Epoch 5 | Loss 0.6173
Epoch 6 | Loss 0.5769
Epoch 7 | Loss 0.5378
Epoch 8 | Loss 0.4924
Epoch 9 | Loss 0.4668
Epoch 10 | Loss 0.4439
Epoch 11 | Loss 0.4229
Epoch 12 | Loss 0.3872
Epoch 13 | Loss 0.3742
Epoch 14 | Loss 0.3549
Epoch 15 | Loss 0.3281


In [ ]:
all_X = []
all_vids = []

for fname in os.listdir(EMB_DIR):
    if not fname.endswith(".npy"):
        continue

    vid = fname.replace(".npy", "")
    emb = np.load(os.path.join(EMB_DIR, fname))

    all_X.append(emb)
    all_vids.append(vid)

all_X = torch.tensor(np.stack(all_X), dtype=torch.float32).to(device)

model.eval()
with torch.no_grad():
    logits = model(all_X)
    probs = torch.softmax(logits, dim=1)

conf, preds = probs.max(dim=1)

conf = conf.cpu().numpy()
preds = preds.cpu().numpy()

In [ ]:
csv_path = os.path.join(SAVE_PATH, "text_predictions_global.csv")
json_path = os.path.join(SAVE_PATH, "text_predictions_global.json")

pred_labels_str = [INV_LABEL_MAP[p] for p in preds]

results_df = pd.DataFrame({
    "video_id": all_vids,
    "predicted_label": preds,
    "predicted_label_str": pred_labels_str,
    "confidence": conf
})

results_df.to_csv(csv_path, index=False)
print("Saved CSV")

results_list = [
    {
        "video_id": vid,
        "predicted_label": int(pred),
        "predicted_label_str": str(pred_str),
        "confidence": float(conf_val)
    }
    for vid, pred, pred_str, conf_val in zip(all_vids, preds, pred_labels_str, conf)
]

with open(json_path, "w") as f:
    json.dump(results_list, f, indent=2)

print("Saved JSON")